# Late Fusion: CLIP + ViLT — FakeNewsNet 2-Way Fake News Detection
**Architecture:** CLIP (text + image encoders) and ViLT (cross-modal transformer) each produce a 512-d feature vector.  
Their outputs are **concatenated** → `[B, 1024]` and passed through a two-layer MLP classifier.  
**Dataset:** FakeNewsNet (GossipCop + PolitiFact) — binary: `real=0 / fake=1`  
**Training:** 5-Fold Stratified CV | 80/10/10 split | 10 epochs | frozen backbones | AMP


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Uncomment on first run if transformers/datasets are not pre-installed
# !pip install -q transformers datasets Pillow scikit-learn tqdm


In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os, random, warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import (
    CLIPProcessor, CLIPModel,
    ViltProcessor, ViltModel,
)
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report,
)
warnings.filterwarnings("ignore")


In [ ]:
# ── Cell 3: Config & Reproducibility ─────────────────────────────────────────
SEED             = 42
NUM_CLASSES      = 2        # 0 = real, 1 = fake
NUM_EPOCHS       = 10
BATCH_SIZE       = 32       # larger batch safe because backbones are frozen
LR               = 2e-4    # head-only LR
N_FOLDS          = 5
MAX_SAMPLES      = 5000     # cap per class (balanced)
FREEZE_BACKBONES = True     # freeze CLIP & ViLT backbone weights
USE_AMP          = True     # mixed-precision training

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# ── Cell 4: Load FakeNewsNet ──────────────────────────────────────────────────
# Supports two common Kaggle dataset layouts:
#
#   Layout A – per-article JSON folders (original KDD FakeNewsNet):
#     /kaggle/input/<ds>/gossipcop/fake/<id>/news content.json
#     /kaggle/input/<ds>/gossipcop/real/<id>/news content.json
#
#   Layout B – flat CSV files (common Kaggle re-uploads):
#     Each CSV path must contain 'fake' or 'real' in its path/filename so
#     the label can be inferred automatically.
#
# Internet must be ON (Notebook settings → Internet → ON) for image downloads.

import glob as _glob
import urllib.request as _urlreq
from concurrent.futures import ThreadPoolExecutor, as_completed as _as_completed
from tqdm import tqdm as _tqdm

IMG_DIR   = "/kaggle/working/images"
_HEADERS  = {"User-Agent": "Mozilla/5.0"}
_TIMEOUT  = 10
_IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

os.makedirs(IMG_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────
def _infer_label(path_str):
    """Return 1 (fake) or 0 (real) from a file/folder path, or None."""
    p = path_str.lower().replace("\\\\", "/")
    if "/fake" in p or "_fake" in p:
        return 1
    if "/real" in p or "_real" in p:
        return 0
    return None

def _find_first_local_image(article_dir):
    img_folder = os.path.join(article_dir, "images")
    if not os.path.isdir(img_folder):
        return None
    for fname in sorted(os.listdir(img_folder)):
        if os.path.splitext(fname)[1].lower() in _IMG_EXTS:
            fpath = os.path.join(img_folder, fname)
            if os.path.getsize(fpath) > 500:
                return fpath
    return None

def _download_image(row_id, url):
    """Download url → IMG_DIR/<row_id>.jpg. Returns local path or None."""
    dest = os.path.join(IMG_DIR, f"{row_id}.jpg")
    if os.path.isfile(dest) and os.path.getsize(dest) > 500:
        return dest
    try:
        req  = _urlreq.Request(str(url), headers=_HEADERS)
        data = _urlreq.urlopen(req, timeout=_TIMEOUT).read()
        if len(data) < 500:
            return None
        with open(dest, "wb") as fh:
            fh.write(data)
        return dest
    except Exception:
        return None

# ── Layout A: JSON folders ────────────────────────────────────────────────────
_json_hits  = _glob.glob("/kaggle/input/**/news content.json", recursive=True)
_json_hits += _glob.glob("/kaggle/input/**/news_content.json", recursive=True)

records = []

if _json_hits:
    print(f"Layout A detected: {len(_json_hits)} article JSON files")
    for _jpath in _tqdm(_json_hits, desc="Reading JSONs"):
        _label = _infer_label(_jpath)
        if _label is None:
            continue
        try:
            with open(_jpath, encoding="utf-8", errors="ignore") as fh:
                _d = json.load(fh)
            _title = str(_d.get("title", "")).strip()
            _body  = str(_d.get("text",  "")).strip()[:256]
            _text  = (_title + " " + _body).strip() if _body else _title
            if not _text:
                continue
            _art_dir  = os.path.dirname(_jpath)
            _img_path = _find_first_local_image(_art_dir)
            _img_url  = str(_d.get("top_img") or _d.get("image_url") or "").strip()
            records.append({
                "id":        os.path.basename(_art_dir),
                "text":      _text,
                "img_path":  _img_path,
                "img_url":   _img_url,
                "label_enc": _label,
            })
        except Exception:
            continue
    print(f"  Loaded {len(records)} articles from JSON layout")

# ── Layout B: CSV files ───────────────────────────────────────────────────────
if not records:
    _csv_hits = _glob.glob("/kaggle/input/**/*.csv", recursive=True)
    _csv_hits = [p for p in _csv_hits
                 if any(tok in p.lower()
                        for tok in ["fake", "real", "gossipcop", "politifact"])]
    if not _csv_hits:
        raise FileNotFoundError(
            "No FakeNewsNet data found under /kaggle/input/.\n"
            "Add the dataset: Notebook → Add data → search 'FakeNewsNet'."
        )
    print(f"Layout B detected: {len(_csv_hits)} CSV file(s)")

    for _cpath in _csv_hits:
        _label = _infer_label(_cpath)
        if _label is None:
            print(f"  Skipping (cannot infer label from path): {_cpath}")
            continue

        try:
            _tmp = pd.read_csv(_cpath, low_memory=False)
        except Exception as e:
            print(f"  Skipping (read error): {_cpath} — {e}")
            continue

        print(f"  {_cpath}  →  label={_label}  cols={_tmp.columns.tolist()[:8]}")

        # ── normalise text column ─────────────────────────────────────────────
        _text_col = None
        for _tc in ["title", "clean_title", "headline", "text", "content"]:
            if _tc in _tmp.columns:
                _text_col = _tc
                break
        if _text_col is None:
            print(f"  Skipping (no text column): {_cpath}")
            continue
        _tmp = _tmp.rename(columns={_text_col: "text"})

        # ── normalise image-URL column (optional) ─────────────────────────────
        _url_col = None
        for _ic in ["top_img", "image_url", "img_url", "images", "image"]:
            if _ic in _tmp.columns:
                _url_col = _ic
                break
        if _url_col is not None:
            _tmp = _tmp.rename(columns={_url_col: "img_url"})
        else:
            _tmp["img_url"] = ""          # no image URL column in this CSV

        # ── ensure id column ──────────────────────────────────────────────────
        if "id" not in _tmp.columns:
            _tmp["id"] = _tmp.index.astype(str) + f"_{_label}"

        # ── clean & type-cast ─────────────────────────────────────────────────
        _tmp = _tmp.dropna(subset=["text"])
        _tmp["text"]      = _tmp["text"].astype(str).str.strip()
        _tmp["img_url"]   = _tmp["img_url"].fillna("").astype(str).str.strip()
        _tmp["img_path"]  = None
        _tmp["label_enc"] = int(_label)
        _tmp = _tmp[_tmp["text"].str.len() > 0]

        records += _tmp[["id", "text", "img_url", "img_path", "label_enc"]].to_dict("records")
        print(f"    → {len(_tmp)} usable rows")

    print(f"Total records loaded from CSVs: {len(records)}")

if not records:
    raise RuntimeError(
        "0 records loaded from FakeNewsNet.\n"
        "Check that your CSV files have a text/title column and that\n"
        "their paths contain 'fake' or 'real' to infer the label."
    )

df = pd.DataFrame(records)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"\nTotal usable articles: {len(df)}  |  label dist: {df['label_enc'].value_counts().to_dict()}")

# ── Balanced sample (4× buffer to absorb download failures) ──────────────────
_min_cls  = df["label_enc"].value_counts().min()
_per_dl   = min(_min_cls, (MAX_SAMPLES // 2) * 4)
df_pool   = pd.concat([
    df[df["label_enc"] == c].sample(
        min(_per_dl, int((df["label_enc"] == c).sum())),
        random_state=SEED
    )
    for c in [0, 1]
]).reset_index(drop=True)

# ── Download images for rows that have a URL but no local path ────────────────
_needs_dl = df_pool[
    df_pool["img_path"].isna() &
    df_pool["img_url"].str.startswith("http", na=False)
]
if len(_needs_dl):
    print(f"Downloading {len(_needs_dl)} images…")
    with ThreadPoolExecutor(max_workers=32) as _pool:
        _futs = {
            _pool.submit(_download_image, row["id"], row["img_url"]): idx
            for idx, row in _needs_dl.iterrows()
        }
        for _fut in _tqdm(_as_completed(_futs), total=len(_futs), desc="Images", unit="img"):
            _idx    = _futs[_fut]
            _result = _fut.result()
            if _result:
                df_pool.at[_idx, "img_path"] = _result
else:
    print("No image URLs to download (images are local or unavailable).")

print(f"Articles with image: {df_pool['img_path'].notna().sum()} / {len(df_pool)}")

# ── Final balanced sample ─────────────────────────────────────────────────────
_per_class = min(df_pool["label_enc"].value_counts().min(), MAX_SAMPLES // 2)
df = pd.concat([
    df_pool[df_pool["label_enc"] == c].sample(_per_class, random_state=SEED)
    for c in [0, 1]
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Final dataset: {len(df)} rows  |  class dist: {df['label_enc'].value_counts().to_dict()}")
print(f"With image: {df['img_path'].notna().sum()}  |  Text-only: {df['img_path'].isna().sum()}")


In [ ]:
# ── Cell 5: Dataset & Collate ─────────────────────────────────────────────────
_GREY = Image.new("RGB", (224, 224), (128, 128, 128))   # shared placeholder

class FakeNewsNetDataset(Dataset):
    """
    Returns (clip_inputs, vilt_inputs, label) for each article.

    Text  : article title / body (truncated to processor limits)
    Image : local file if available, else grey 224×224 placeholder
    Label : 0 = real, 1 = fake
    """
    def __init__(self, df, clip_processor, vilt_processor):
        self.df             = df.reset_index(drop=True)
        self.clip_processor = clip_processor
        self.vilt_processor = vilt_processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("text") or row.get("title") or "")[:512]
        lbl  = int(row["label_enc"])

        # ── Image ─────────────────────────────────────────────────────────────
        img_path = row.get("img_path")
        try:
            if img_path and os.path.isfile(str(img_path)):
                image = Image.open(img_path).convert("RGB").resize((224, 224))
            else:
                image = _GREY.copy()
        except Exception:
            image = _GREY.copy()

        # ── CLIP inputs (max 77 tokens — CLIP's native positional limit) ──────
        clip_in = self.clip_processor(
            text=[text], images=image,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=77,
        )
        clip_in = {k: v.squeeze(0) for k, v in clip_in.items()}

        # ── ViLT inputs (capped at 40 to stay within ViLT's limit) ───────────
        vilt_in = self.vilt_processor(
            text=[text], images=image,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=40,
        )
        vilt_in = {k: v.squeeze(0) for k, v in vilt_in.items()}

        return clip_in, vilt_in, lbl


def collate_fn(batch):
    clip_batch, vilt_batch, labels = zip(*batch)

    def stack(lst):
        return {k: torch.stack([d[k] for d in lst]) for k in lst[0].keys()}

    return stack(clip_batch), stack(vilt_batch), torch.tensor(labels, dtype=torch.long)


In [ ]:
# ── Cell 6: Late Fusion Model ─────────────────────────────────────────────────
#
# Late Fusion strategy
# ─────────────────────
# Both backbones run independently on the same (text, image) pair.
#
#   CLIP backbone
#     text_model  → pooler → text_projection  → L2-norm  → text_emb   [B, 512]
#     vision_model → pooler → visual_projection → L2-norm → image_emb  [B, 512]
#     concat([text_emb, image_emb])                                     [B,1024]
#     fc(Linear → GELU → Dropout)                           clip_feat  [B, 512]
#
#   ViLT backbone (internal cross-modal attention across text+image tokens)
#     CLS token of last hidden state                                    [B, 768]
#     fc(Linear → GELU → Dropout)                           vilt_feat  [B, 512]
#
#   Late Fusion
#     concat([clip_feat, vilt_feat])                          fused     [B,1024]
#     classifier(Linear→GELU→Dropout→Linear→GELU→Dropout→Linear)       [B,  2]
#
# Backbones are frozen; only the two FC projection layers and the
# classifier head are trained (fast, GPU-friendly, minimal overfitting).

class CLIPFeatureExtractor(nn.Module):
    """Extract a joint text+image embedding from CLIP, project to 512-d."""
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        d = clip_model.config.projection_dim          # 512 for ViT-B/32
        self.fc = nn.Sequential(
            nn.Linear(d * 2, 512),
            nn.GELU(),
            nn.Dropout(0.3),
        )

    def forward(self, clip_inputs):
        txt = self.clip.text_model(
            input_ids      = clip_inputs["input_ids"],
            attention_mask = clip_inputs["attention_mask"],
        )
        vis = self.clip.vision_model(
            pixel_values   = clip_inputs["pixel_values"],
        )
        t = F.normalize(self.clip.text_projection(txt.pooler_output),    dim=-1)  # [B,512]
        v = F.normalize(self.clip.visual_projection(vis.pooler_output),  dim=-1)  # [B,512]
        return self.fc(torch.cat([t, v], dim=-1))                                  # [B,512]


class ViLTFeatureExtractor(nn.Module):
    """Extract the CLS cross-modal token from ViLT, project to 512-d."""
    def __init__(self, vilt_model):
        super().__init__()
        self.vilt = vilt_model
        d = vilt_model.config.hidden_size              # 768 for vilt-b32
        self.fc = nn.Sequential(
            nn.Linear(d, 512),
            nn.GELU(),
            nn.Dropout(0.3),
        )

    def forward(self, vilt_inputs):
        out = self.vilt(
            input_ids      = vilt_inputs["input_ids"],
            attention_mask = vilt_inputs["attention_mask"],
            token_type_ids = vilt_inputs.get("token_type_ids"),
            pixel_values   = vilt_inputs["pixel_values"],
            pixel_mask     = vilt_inputs.get("pixel_mask"),
        )
        cls = out.last_hidden_state[:, 0, :]           # [B, 768]
        return self.fc(cls)                             # [B, 512]


class LateFusionClassifier(nn.Module):
    """
    Full Late Fusion model:
        clip_feat  [B, 512]  ─┐
                               ├─ concat [B,1024] ─► classifier ─► logits [B,2]
        vilt_feat  [B, 512]  ─┘
    """
    def __init__(self, clip_model, vilt_model, num_classes=2, freeze_backbones=True):
        super().__init__()
        self.clip_fe = CLIPFeatureExtractor(clip_model)
        self.vilt_fe = ViLTFeatureExtractor(vilt_model)

        self.classifier = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

        if freeze_backbones:
            for p in self.clip_fe.clip.parameters():
                p.requires_grad = False
            for p in self.vilt_fe.vilt.parameters():
                p.requires_grad = False
        # FC projections + classifier are always trainable

    def forward(self, clip_inputs, vilt_inputs):
        clip_feat = self.clip_fe(clip_inputs)                       # [B, 512]
        vilt_feat = self.vilt_fe(vilt_inputs)                       # [B, 512]
        fused     = torch.cat([clip_feat, vilt_feat], dim=-1)       # [B,1024]
        return self.classifier(fused)                               # [B,   2]


In [ ]:
# ── Cell 7: Training & Evaluation Helpers ────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []

    for clip_in, vilt_in, labels in loader:
        clip_in = {k: v.to(DEVICE) for k, v in clip_in.items()}
        vilt_in = {k: v.to(DEVICE) for k, v in vilt_in.items()}
        labels  = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(clip_in, vilt_in)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        preds_all  += logits.argmax(dim=1).cpu().tolist()
        labels_all += labels.cpu().tolist()

    acc = accuracy_score(labels_all, preds_all)
    return total_loss / len(labels_all), acc


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []

    for clip_in, vilt_in, labels in loader:
        clip_in = {k: v.to(DEVICE) for k, v in clip_in.items()}
        vilt_in = {k: v.to(DEVICE) for k, v in vilt_in.items()}
        labels  = labels.to(DEVICE)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(clip_in, vilt_in)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds_all  += logits.argmax(dim=1).cpu().tolist()
        labels_all += labels.cpu().tolist()

    n  = len(labels_all)
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average="binary", zero_division=0)
    pr  = precision_score(labels_all, preds_all, average="binary", zero_division=0)
    rc  = recall_score(labels_all, preds_all, average="binary", zero_division=0)
    return total_loss / n, acc, f1, pr, rc, preds_all, labels_all


In [ ]:
# ── Cell 8: 5-Fold Cross-Validation ──────────────────────────────────────────
print("\n📥 Loading processors…")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
vilt_processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr   = df["label_enc"].values
fold_results = []
CLASS_NAMES  = ["real", "fake"]

print("=" * 70)
print(f"  Late Fusion CLIP + ViLT  |  FakeNewsNet 2-Way  |  {N_FOLDS}-Fold CV")
print("=" * 70)

full_ds = FakeNewsNetDataset(df, clip_processor, vilt_processor)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df, labels_arr)):
    print(f"\n{'─'*70}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*70}")

    # ── 80 / 10 / 10 split ────────────────────────────────────────────────────
    # train_val_idx holds 90% of data; we carve 10% of *total* as val.
    rng       = np.random.default_rng(SEED + fold)
    perm      = rng.permutation(len(train_val_idx))
    val_cut   = int(0.1 * len(df))               # 10% of total = val
    val_rel   = perm[:val_cut]                    # indices into train_val_idx
    train_rel = perm[val_cut:]

    train_idx = train_val_idx[train_rel]
    val_idx   = train_val_idx[val_rel]

    print(f"  Train: {len(train_idx)}  |  Val: {len(val_idx)}  |  Test: {len(test_idx)}")

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_loader = DataLoader(
        Subset(full_ds, train_idx),
        batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        Subset(full_ds, val_idx),
        batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    test_loader = DataLoader(
        Subset(full_ds, test_idx),
        batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    clip_base = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    vilt_base = ViltModel.from_pretrained("dandelin/vilt-b32-mlm")
    model = LateFusionClassifier(
        clip_base, vilt_base,
        num_classes=NUM_CLASSES,
        freeze_backbones=FREEZE_BACKBONES,
    ).to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"  Params: {total_p/1e6:.1f}M total  |  {trainable/1e3:.1f}K trainable")

    # ── Loss: class-weighted + label smoothing ────────────────────────────────
    counts  = np.bincount(df.iloc[train_idx]["label_enc"].values, minlength=NUM_CLASSES)
    weights = (1.0 / (counts + 1e-6))
    weights = weights / weights.sum() * NUM_CLASSES
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE),
        label_smoothing=0.1,
    )

    # ── Optimiser + cosine LR scheduler ──────────────────────────────────────
    total_steps = len(train_loader) * NUM_EPOCHS
    optimizer   = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-2,
    )
    scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
    scaler      = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_acc = 0.0
    best_state   = None

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc                              = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
        vl_loss, vl_acc, vl_f1, vl_pr, vl_rc, _, _ = eval_epoch(model, val_loader,   criterion)
        gap = round(tr_acc - vl_acc, 4)

        print(f"  Ep {epoch:02d}/{NUM_EPOCHS}  "
              f"TR-Loss={tr_loss:.4f}  TR-Acc={tr_acc:.4f}  |  "
              f"VL-Loss={vl_loss:.4f}  VL-Acc={vl_acc:.4f}  "
              f"F1={vl_f1:.4f}  Gap={gap:+.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # ── Test evaluation (best checkpoint) ────────────────────────────────────
    model.load_state_dict(best_state)
    model.to(DEVICE)
    _, ts_acc, ts_f1, ts_pr, ts_rc, ts_preds, ts_labels = eval_epoch(model, test_loader, criterion)
    # compute final overfitting gap from last epoch training acc
    final_gap = round(tr_acc - vl_acc, 4)

    print(f"\n  ✅ Fold {fold+1} TEST  "
          f"Acc={ts_acc:.4f}  F1={ts_f1:.4f}  "
          f"Prec={ts_pr:.4f}  Rec={ts_rc:.4f}  Gap={final_gap:+.4f}")
    print("\n  Classification Report:\n",
          classification_report(ts_labels, ts_preds,
                                target_names=CLASS_NAMES, zero_division=0))

    fold_results.append({
        "Fold":         fold + 1,
        "Test Acc":     round(ts_acc, 4),
        "Test F1":      round(ts_f1,  4),
        "Precision":    round(ts_pr,  4),
        "Recall":       round(ts_rc,  4),
        "Best Val Acc": round(best_val_acc, 4),
        "Overfit Gap":  round(final_gap, 4),
    })

    del model, clip_base, vilt_base
    torch.cuda.empty_cache()


In [ ]:
# ── Cell 9: Results Summary ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  FINAL 5-FOLD RESULTS — Late Fusion CLIP + ViLT | FakeNewsNet 2-Way")
print("=" * 70)

results_df = pd.DataFrame(fold_results)
mean_row   = results_df.drop(columns=["Fold"]).mean().round(4).to_dict(); mean_row["Fold"] = "Mean"
std_row    = results_df.drop(columns=["Fold"]).std().round(4).to_dict();  std_row["Fold"]  = "Std"
summary_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print(summary_df.to_string(index=False))
summary_df.to_csv("/kaggle/working/late_fusion_fold_results.csv", index=False)
print("\n📄 Results saved to /kaggle/working/late_fusion_fold_results.csv")

try:
    from IPython.display import display
    display(
        summary_df.style
        .set_caption("Late Fusion CLIP+ViLT — FakeNewsNet 5-Fold 2-Way Results")
        .background_gradient(subset=["Test Acc","Test F1","Precision","Recall"], cmap="YlGn")
        .applymap(
            lambda v: "color:red"   if isinstance(v, float) and v >  0.05
               else ("color:green" if isinstance(v, float) and v <= 0.02 else ""),
            subset=["Overfit Gap"]
        )
        .format(precision=4)
    )
except Exception:
    pass

mean_acc = results_df["Test Acc"].mean()
mean_f1  = results_df["Test F1"].mean()
print(f"\n🏆  Mean Test Accuracy : {mean_acc:.4f}  ({mean_acc*100:.2f}%)")
print(f"🏆  Mean Test F1-Score  : {mean_f1:.4f}  ({mean_f1*100:.2f}%)")
